In [1]:
!pip install -q torch

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import re, random, unicodedata
from collections import Counter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
!wget -q "https://www.manythings.org/anki/rus-eng.zip" -O rus-eng.zip
!unzip -q rus-eng.zip
print("Dataset ready: rus.txt")

Dataset ready: rus.txt


In [3]:

MAX_LEN   = 15
MAX_PAIRS = 50000

def normalize(text):
    text = text.lower().strip()
    text = ''.join(c for c in unicodedata.normalize('NFD', text)
                   if unicodedata.category(c) != 'Mn')
    text = re.sub(r"[^a-zA-Zа-яА-ЯёЁ\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def load_pairs(path, max_pairs=MAX_PAIRS):
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2: continue
            en, ru = normalize(parts[0]), normalize(parts[1])
            if len(en.split()) <= MAX_LEN and len(ru.split()) <= MAX_LEN:
                pairs.append((en, ru))
            if len(pairs) >= max_pairs:
                break
    random.shuffle(pairs)
    return pairs

pairs = load_pairs("rus.txt")
print(f"Loaded {len(pairs)} pairs")
print("Sample:", pairs[0])

# --- Build vocabularies ---
class Vocab:
    PAD, SOS, EOS, UNK = 0, 1, 2, 3
    def __init__(self, name):
        self.name = name
        self.word2idx = {'<pad>':0,'<sos>':1,'<eos>':2,'<unk>':3}
        self.idx2word = {0:'<pad>',1:'<sos>',2:'<eos>',3:'<unk>'}

    def build(self, sentences, min_freq=2):
        counts = Counter(w for s in sentences for w in s.split())
        for word, freq in counts.items():
            if freq >= min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx]  = word

    def encode(self, sentence):
        return [self.word2idx.get(w, self.UNK) for w in sentence.split()]

    def __len__(self): return len(self.word2idx)

split = int(0.9 * len(pairs))
train_pairs, val_pairs = pairs[:split], pairs[split:]

en_vocab = Vocab("en"); ru_vocab = Vocab("ru")
en_vocab.build([p[0] for p in train_pairs])
ru_vocab.build([p[1] for p in train_pairs])
print(f"EN vocab: {len(en_vocab)}, RU vocab: {len(ru_vocab)}")

Loaded 50000 pairs
Sample: ('here is my proof', 'вот мое доказательство')
EN vocab: 3335, RU vocab: 7152


In [4]:

class TranslationDataset(Dataset):
    def __init__(self, pairs, en_vocab, ru_vocab):
        self.data = pairs
        self.en_v = en_vocab
        self.ru_v = ru_vocab

    def __len__(self): return len(self.data)

    def __getitem__(self, i):
        en, ru = self.data[i]
        src = torch.tensor(self.en_v.encode(en), dtype=torch.long)
        trg = torch.tensor([Vocab.SOS] + self.ru_v.encode(ru) + [Vocab.EOS], dtype=torch.long)
        return src, trg

def collate_fn(batch):
    srcs, trgs = zip(*batch)
    srcs = pad_sequence(srcs, batch_first=True, padding_value=Vocab.PAD)
    trgs = pad_sequence(trgs, batch_first=True, padding_value=Vocab.PAD)
    return srcs, trgs

BATCH = 128
train_loader = DataLoader(TranslationDataset(train_pairs, en_vocab, ru_vocab),
                          batch_size=BATCH, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(TranslationDataset(val_pairs,   en_vocab, ru_vocab),
                          batch_size=BATCH, collate_fn=collate_fn)
print("DataLoaders ready.")

DataLoaders ready.


In [5]:

EMBED_DIM  = 256
HIDDEN_DIM = 1024

class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=Vocab.PAD)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True)

    def forward(self, src):
        _, (h, c) = self.lstm(self.emb(src))
        return h, c   # (1, B, H)


class Decoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=Vocab.PAD)
        self.lstm = nn.LSTM(EMBED_DIM, HIDDEN_DIM, batch_first=True)
        self.fc   = nn.Linear(HIDDEN_DIM, vocab_size)

    def forward(self, token, h, c):
        # token: (B,) → (B,1,E)
        out, (h, c) = self.lstm(self.emb(token.unsqueeze(1)), (h, c))
        logits = self.fc(out.squeeze(1))   # (B, V)
        return logits, h, c


class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, ru_vocab_size, teacher_ratio=0.5):
        super().__init__()
        self.encoder       = encoder
        self.decoder       = decoder
        self.ru_vocab_size = ru_vocab_size
        self.teacher_ratio = teacher_ratio

    def forward(self, src, trg):
        B, T     = trg.shape
        outputs  = torch.zeros(B, T, self.ru_vocab_size).to(src.device)
        h, c     = self.encoder(src)
        token    = trg[:, 0]       # <sos>
        for t in range(1, T):
            logits, h, c = self.decoder(token, h, c)
            outputs[:, t] = logits
            token = trg[:, t] if random.random() < self.teacher_ratio \
                    else logits.argmax(1)
        return outputs

encoder = Encoder(len(en_vocab))
decoder = Decoder(len(ru_vocab))
model   = Seq2Seq(encoder, decoder, len(ru_vocab)).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 20,517,616


In [6]:
# Block 6: Train & Evaluate

LR      = 0.001
EPOCHS  = 10
CLIP    = 1.0

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss(ignore_index=Vocab.PAD)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            # During eval: teacher_ratio=0 (no teacher forcing)
            if not train: model.teacher_ratio = 0
            out = model(src, trg)          # (B, T, V)
            loss = criterion(out[:, 1:].reshape(-1, len(ru_vocab)),
                             trg[:, 1:].reshape(-1))
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                optimizer.step()
            total_loss += loss.item()
    model.teacher_ratio = 0.5             # restore
    return total_loss / len(loader)

best_val = float('inf')
for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss   = run_epoch(val_loader,   train=False)
    marker = " ← saved" if val_loss < best_val else ""
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), "best_model.pt")
    print(f"Epoch {epoch:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f}{marker}")

Epoch 01 | Train: 3.7764 | Val: 2.6217 ← saved
Epoch 02 | Train: 2.0582 | Val: 1.8350 ← saved
Epoch 03 | Train: 1.2819 | Val: 1.6424 ← saved
Epoch 04 | Train: 0.9661 | Val: 1.6088 ← saved
Epoch 05 | Train: 0.8274 | Val: 1.5803 ← saved
Epoch 06 | Train: 0.7497 | Val: 1.6142
Epoch 07 | Train: 0.7090 | Val: 1.6176
Epoch 08 | Train: 0.6667 | Val: 1.6185
Epoch 09 | Train: 0.6353 | Val: 1.6473
Epoch 10 | Train: 0.6235 | Val: 1.6464


In [7]:
# Block 7: Translate

model.load_state_dict(torch.load("best_model.pt"))
model.eval()

def translate(sentence, max_len=50):
    sentence = normalize(sentence)
    src = torch.tensor(en_vocab.encode(sentence), dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        h, c  = model.encoder(src)
        token = torch.tensor([Vocab.SOS], dtype=torch.long).to(device)
        result = []
        for _ in range(max_len):
            logits, h, c = model.decoder(token, h, c)
            token = logits.argmax(1)
            word  = ru_vocab.idx2word[token.item()]
            if word == '<eos>': break
            result.append(word)
    return ' '.join(result)

# Test it
test_sentences = ["hello", "how are you", "i love you", "good morning"]
for s in test_sentences:
    print(f"EN: {s:20s} → RU: {translate(s)}")

EN: hello                → RU: <unk>
EN: how are you          → RU: как ты
EN: i love you           → RU: я тебя люблю
EN: good morning         → RU: доброе утро
